# tui

> Terminal live inspector — a tight, gruvbox-dark Textual frontend that runs in
> its own terminal and streams the live kernel namespace from the in-kernel paar
> server (the same one `paar.fasthtml.inspector()` starts), the way an iframe or
> `htop` sits live in a pane.

In [ ]:
#| default_exp tui

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import asyncio, json, argparse
from urllib.parse import quote
import httpx

# Gruvbox-dark palette (https://github.com/morhetz/gruvbox) — the whole UI is keyed off these.
GRUV = dict(
    bg_h='#1d2021', bg='#282828', bg1='#3c3836', bg2='#504945', bg3='#665c54', bg4='#7c6f64',
    fg='#ebdbb2', fg2='#d5c4a1', fg4='#a89984', gray='#928374',
    red='#fb4934', green='#b8bb26', yellow='#fabd2f', blue='#83a598',
    purple='#d3869b', aqua='#8ec07c', orange='#fe8019')

def _acc(accessor):
    "URL-safe positional accessor, matching the web frontend's encoding."
    return quote(json.dumps(list(accessor)), safe='')

def _ws_url(base):
    "http(s) base -> ws(s) URL for the /live nudge socket."
    b = base.rstrip('/')
    if b.startswith('https'): return 'wss' + b[5:] + '/live'
    if b.startswith('http'):  return 'ws'  + b[4:] + '/live'
    return b + '/live'

class Client:
    "Async HTTP+WS client for a running in-kernel paar server (see paar.fasthtml)."
    def __init__(self, base='http://127.0.0.1:8000'):
        self.base = base.rstrip('/')
        self.ws_url = _ws_url(self.base)
        self._http = httpx.AsyncClient(base_url=self.base, timeout=10.0)
    async def rows(self, profile=None):
        r = await self._http.get('/api/rows', params={'profile': profile} if profile else None)
        r.raise_for_status(); return r.json()
    async def expand(self, accessor, offset=0):
        r = await self._http.get(f'/api/expand?accessor={_acc(accessor)}&offset={offset}')
        r.raise_for_status(); return r.json()
    async def grid(self, accessor, roff=0, coff=0):
        r = await self._http.get(f'/api/grid?accessor={_acc(accessor)}&roff={roff}&coff={coff}')
        r.raise_for_status(); return r.json()
    async def aclose(self): await self._http.aclose()

In [ ]:
#| export
from rich.text import Text

def fmt_row(v, maxval=90):
    "A VarInfo dict -> a single-line gruvbox Rich Text: name {type: shape} value  dtype ▦, mirroring the web row."
    g, err = GRUV, v.get('is_error')
    t = Text(no_wrap=True, overflow='ellipsis')
    t.append(v.get('name',''), style=f"bold {g['red'] if err else g['aqua']}")
    typ = v.get('type') or ''
    if v.get('shape'): typ = f"{typ}: {v['shape']}"
    if typ: t.append('  '); t.append('{'+typ+'}', style=g['gray'])
    val = v.get('value') or ''
    if len(val) > maxval: val = val[:maxval-1] + '\u2026'
    if val: t.append('  '); t.append(val, style=g['red'] if err else g['fg2'])
    if v.get('dtype'): t.append(f"  {v['dtype']}", style=f"italic {g['gray']}")
    if v.get('is_grid'): t.append('  \u25a6', style=g['orange'])   # ▦ = openable grid panel
    return t

def group_label(name):
    "Uppercased dim-yellow heading for a category group (Modules / Functions / …)."
    return Text(str(name).upper(), style=f"bold {GRUV['yellow']}")

In [ ]:
#| export
from textual.theme import Theme

def gruvbox_theme():
    "A tightened gruvbox-dark Textual theme (deep bg, aqua cursor)."
    g = GRUV
    return Theme(
        name='paar-gruvbox', dark=True,
        primary=g['aqua'], secondary=g['blue'], accent=g['orange'],
        warning=g['yellow'], error=g['red'], success=g['green'],
        foreground=g['fg'], background=g['bg_h'], surface=g['bg'], panel=g['bg1'],
        variables={'block-cursor-foreground': g['bg_h'], 'block-cursor-background': g['aqua'],
                   'block-cursor-text-style': 'none', 'footer-key-foreground': g['aqua'],
                   'scrollbar': g['bg2'], 'scrollbar-hover': g['bg3']})

In [ ]:
#| export
from textual.app import App, ComposeResult
from textual.binding import Binding
from textual.containers import Vertical
from textual.widgets import Tree, DataTable, Static, Footer

_PAGE = 50   # grid page window (rows/cols) per step

class InspectorApp(App):
    "Live terminal inspector: a lazy gruvbox tree of the kernel namespace with a paged grid panel, refreshed on every cell run."
    CSS = """
    Screen { background: $background; }
    #topbar { height: 1; background: $panel; color: $foreground; padding: 0 1; }
    #tree { padding: 0 0 0 1; background: $background; scrollbar-size-vertical: 1; }
    #tree > .tree--guides          { color: $surface; }
    #tree > .tree--guides-hover    { color: $panel; }
    #tree > .tree--guides-selected { color: $accent; }
    #gridwrap { display: none; height: 45%; }
    #gridwrap.visible { display: block; }
    #gridnav { height: 1; background: $panel; color: $foreground; padding: 0 1; }
    #grid { background: $background; scrollbar-size: 1 1; }
    Footer { background: $panel; }
    Footer > .footer-key--description { color: $foreground; }
    """
    BINDINGS = [
        Binding('r', 'refresh', 'refresh'),
        Binding('g', 'toggle_grid', 'grid'),
        Binding('escape', 'hide_grid', 'close', show=False),
        Binding('1', "profile('minimal')", 'min'),
        Binding('2', "profile('standard')", 'std'),
        Binding('3', "profile('full')", 'full'),
        Binding('bracketright', "grid_page('rows', 1)", 'rows+', show=False),
        Binding('bracketleft',  "grid_page('rows', -1)", 'rows-', show=False),
        Binding('right_curly_bracket', "grid_page('cols', 1)", 'cols+', show=False),
        Binding('left_curly_bracket',  "grid_page('cols', -1)", 'cols-', show=False),
        Binding('q', 'quit', 'quit'),
    ]

    def __init__(self, base='http://127.0.0.1:8000', profile='standard', client=None, connect_ws=True):
        super().__init__()
        self.client = client or Client(base)
        self.profile = profile
        self.connect_ws = connect_ws
        self._by_acc = {}      # tuple(accessor) -> TreeNode  (data nodes only)
        self._loaded = set()   # accessors whose children have been fetched
        self._live = False
        self._grid = None      # [accessor, roff, coff] while the grid panel is open

    # -- layout ---------------------------------------------------------------
    def compose(self) -> ComposeResult:
        yield Static(id='topbar')
        self._tree = Tree('paar', id='tree')
        self._tree.show_root = False
        self._tree.guide_depth = 2
        yield self._tree
        with Vertical(id='gridwrap'):
            yield Static(id='gridnav')
            yield DataTable(id='grid', zebra_stripes=True, cursor_type='row', show_row_labels=True)
        yield Footer()

    async def on_mount(self):
        self.register_theme(gruvbox_theme()); self.theme = 'paar-gruvbox'
        await self._reload()
        if self.connect_ws: self.run_worker(self._ws_loop(), name='ws', exclusive=True)

    # -- tree building --------------------------------------------------------
    def _add_node(self, parent, v):
        acc = tuple(v['accessor'])
        kind = 'more' if v.get('more_offset') is not None else 'node'
        expandable = kind == 'node' and bool(v.get('is_container'))
        node = parent.add(fmt_row(v), data={'kind': kind, 'v': v, 'acc': acc},
                          allow_expand=expandable)
        if kind == 'node': self._by_acc[acc] = node
        return node

    def _build(self, data):
        self._tree.clear(); self._by_acc.clear(); self._loaded.clear()
        root = self._tree.root
        for grp in data['groups']:
            if grp['label'] is None:
                for v in grp['nodes']: self._add_node(root, v)
            else:
                gnode = root.add(group_label(grp['label']), data={'kind': 'group'})
                for v in grp['nodes']: self._add_node(gnode, v)
                gnode.expand()

    async def _load(self, node, offset=0):
        "Fetch and append one page of a container node's children."
        for v in await self.client.expand(node.data['acc'], offset):
            self._add_node(node, v)
        self._loaded.add(node.data['acc'])

    def _cursor_acc(self):
        n = self._tree.cursor_node
        return (n.data or {}).get('acc') if n and n.data else None

    async def _reload(self):
        "Re-pull the namespace and rebuild, replaying open subtrees and the cursor (the live htop-style refresh)."
        try:
            data = await self.client.rows(self.profile)
        except Exception as e:
            self._set_live(False); self._topbar(err=str(e)); return
        expanded = sorted((acc for acc, n in self._by_acc.items() if n.is_expanded), key=len)
        cursor = self._cursor_acc()
        self.profile = data.get('profile', self.profile)
        self._build(data)
        for acc in expanded:                      # replay drill-down, shallow first
            node = self._by_acc.get(acc)
            if node is None or not node.allow_expand: continue
            if acc not in self._loaded: await self._load(node)
            node.expand()
        if cursor and cursor in self._by_acc:
            self._tree.move_cursor(self._by_acc[cursor])
        self._topbar()

    # -- events ---------------------------------------------------------------
    async def on_tree_node_expanded(self, event):
        d = event.node.data or {}
        if d.get('kind') == 'node' and d['acc'] not in self._loaded and not event.node.children:
            await self._load(event.node)

    async def on_tree_node_selected(self, event):
        d = event.node.data or {}
        if d.get('kind') == 'more':
            v, parent = d['v'], event.node.parent
            event.node.remove()
            for nv in await self.client.expand(v['accessor'], v['more_offset']):
                self._add_node(parent, nv)
        elif d.get('kind') == 'node' and d['v'].get('is_grid'):
            await self._open_grid(d['v'])

    # -- grid panel -----------------------------------------------------------
    async def _open_grid(self, v):
        self._grid = [list(v['accessor']), 0, 0]
        await self._render_grid()
        self.query_one('#gridwrap').add_class('visible')

    async def _render_grid(self):
        acc, roff, coff = self._grid
        page = await self.client.grid(acc, roff, coff)
        dt = self.query_one('#grid', DataTable); dt.clear(columns=True)
        nav = self.query_one('#gridnav', Static)
        if not page:
            nav.update('not gridable'); return
        for h in page['headers']: dt.add_column(str(h))
        for ix, row in zip(page['index'], page['cells']):
            dt.add_row(*[str(c) for c in row], label=str(ix))
        npr, npc = len(page['index']), len(page['headers'])
        nav.update(f"[b {GRUV['orange']}]\u25a6[/]  rows {roff}\u2013{roff+npr}/{page['nrows']}   "
                   f"cols {coff}\u2013{coff+npc}/{page['ncols']}   "
                   f"[dim][ ] rows   {{ }} cols   esc close[/]")

    async def action_grid_page(self, axis, direction):
        if not self._grid: return
        acc, roff, coff = self._grid
        if axis == 'rows': roff = max(0, roff + direction*_PAGE)
        else:              coff = max(0, coff + direction*_PAGE)
        self._grid = [acc, roff, coff]; await self._render_grid()

    async def action_toggle_grid(self):
        w = self.query_one('#gridwrap')
        if w.has_class('visible'): w.remove_class('visible'); return
        n = self._tree.cursor_node
        if n and (n.data or {}).get('v', {}).get('is_grid'):
            await self._open_grid(n.data['v'])

    def action_hide_grid(self):
        self.query_one('#gridwrap').remove_class('visible')

    async def action_profile(self, name):
        self.profile = name; await self._reload()

    async def action_refresh(self): await self._reload()

    # -- live socket ----------------------------------------------------------
    async def _ws_loop(self):
        "Hold the /live socket open; any frame means a cell ran -> re-pull. Reconnect on drop."
        from websockets.asyncio.client import connect as ws_connect
        while True:
            try:
                async with ws_connect(self.client.ws_url) as ws:
                    self._set_live(True)
                    async for _ in ws: await self._reload()
            except Exception:
                self._set_live(False)
                await asyncio.sleep(1.0)

    # -- top bar --------------------------------------------------------------
    def _set_live(self, on):
        self._live = on; self._topbar()

    def _topbar(self, err=None):
        try: bar = self.query_one('#topbar', Static)
        except Exception: return
        g = GRUV
        if err:
            state = f"[{g['red']}]\u25cb[/] {err[:40]}"
        else:
            state = (f"[{g['green']}]\u25cf[/] live" if self._live
                     else f"[{g['yellow']}]\u25cb[/] connecting\u2026")
        bar.update(f" [b {g['aqua']}]paar[/]  \u00b7  [{g['fg4']}]{self.profile}[/]  \u00b7  {state}"
                   f"    [dim]1/2/3 profile   g grid   r refresh   q quit[/]")

In [ ]:
#| export
def run(url='http://127.0.0.1:8000', profile='standard'):
    "Launch the terminal inspector against a running paar server (start one with paar.fasthtml.inspector())."
    InspectorApp(base=url, profile=profile).run()

def main(argv=None):
    "Console entry point: `paar-tui [--url ...] [--profile ...]`."
    p = argparse.ArgumentParser(prog='paar-tui', description='live terminal namespace inspector')
    p.add_argument('--url', default='http://127.0.0.1:8000', help='paar server base URL')
    p.add_argument('--profile', default='standard', choices=['minimal', 'standard', 'full'])
    a = p.parse_args(argv); run(a.url, a.profile)

In [ ]:
from types import SimpleNamespace

def test_ws_url():
    assert _ws_url('http://127.0.0.1:8000') == 'ws://127.0.0.1:8000/live'
    assert _ws_url('https://h:9/') == 'wss://h:9/live'

def test_acc_roundtrip():
    assert json.loads(__import__('urllib.parse', fromlist=['unquote']).unquote(_acc(('a', 3)))) == ['a', 3]

def test_fmt_row_basic():
    t = fmt_row({'name': 'x', 'type': 'int', 'value': '5', 'accessor': ['x']})
    s = t.plain
    assert 'x' in s and '{int}' in s and '5' in s

def test_fmt_row_shape_dtype_grid():
    t = fmt_row({'name': 'arr', 'type': 'ndarray', 'shape': '(3,)', 'dtype': 'int64',
                 'value': '[1 2 3]', 'is_grid': True, 'accessor': ['arr']})
    s = t.plain
    assert 'ndarray: (3,)' in s and 'int64' in s and '\u25a6' in s   # shape, dtype, grid mark

def test_fmt_row_truncates():
    t = fmt_row({'name': 'big', 'type': 'str', 'value': 'z'*500, 'accessor': ['big']}, maxval=20)
    assert t.plain.endswith('\u2026') and len(t.plain) < 120

def test_theme_is_dark_gruvbox():
    th = gruvbox_theme()
    assert th.dark and th.background == GRUV['bg_h'] and th.primary == GRUV['aqua']

for _t in [test_ws_url, test_acc_roundtrip, test_fmt_row_basic, test_fmt_row_shape_dtype_grid,
           test_fmt_row_truncates, test_theme_is_dark_gruvbox]: _t()

In [ ]:
class _StubClient:
    "In-memory stand-in for Client so the app can be driven with no server."
    ws_url = 'ws://stub/live'
    async def rows(self, profile=None):
        return {'profile': profile or 'standard', 'profiles': ['minimal', 'standard', 'full'],
                'groups': [{'label': None, 'nodes': [
                    {'name': 'alpha', 'type': 'int', 'value': '123', 'accessor': ['alpha'],
                     'is_container': False, 'is_grid': False},
                    {'name': 'd', 'type': 'dict', 'value': '{...}', 'accessor': ['d'],
                     'is_container': True, 'is_grid': False}]},
                    {'label': 'Modules', 'nodes': [
                    {'name': 'mymod', 'type': 'module', 'value': 'math', 'accessor': ['mymod'],
                     'is_container': True, 'is_grid': False}]}]}
    async def expand(self, accessor, offset=0):
        return [{'name': "'x'", 'type': 'int', 'value': '1', 'accessor': list(accessor)+[0],
                 'is_container': False, 'is_grid': False}]
    async def grid(self, accessor, roff=0, coff=0): return None
    async def aclose(self): pass

async def _pilot():
    app = InspectorApp(client=_StubClient(), connect_ws=False)
    async with app.run_test() as pilot:
        labels = [str(n.label) for n in app._tree.root.children]
        assert any('alpha' in l for l in labels), labels          # data node rendered
        assert any('MODULES' in l for l in labels), labels        # group heading rendered
        dnode = app._by_acc[('d',)]
        dnode.expand(); await pilot.pause()
        assert dnode.children, 'container lazily loaded its child on expand'
    await app.client.aclose()

# Textual's run_test needs to own the event loop; run it only when one isn't already spinning
# (i.e. under `nbdev_test`/plain execution, not inside a live Jupyter kernel).
try: asyncio.get_running_loop()
except RuntimeError: asyncio.run(_pilot())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()